# Homework 13 — Productization

**Author:** Paritosh Dwivedi

The last homework of the course, and deliberately self-contained: the data is generated here, so
nothing from a previous week is needed.

Four files in this folder: this notebook, `model/model.pkl`, `app.py`, and `README.md`.

## 1. Generate data and train a model

In [1]:
import json
import os
import threading

import joblib
import numpy as np
import requests
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

X, y = make_regression(n_samples=100, n_features=2, noise=0.1, random_state=42)
model = LinearRegression().fit(X, y)
print(f'trained on {X.shape[0]} samples, {X.shape[1]} features')
print(f'coefficients {model.coef_.round(3)}   intercept {model.intercept_:.3f}')

trained on 100 samples, 2 features
coefficients [87.72  74.077]   intercept 0.002


In [2]:
# Create model/ BEFORE saving, or joblib.dump raises FileNotFoundError.
os.makedirs('model', exist_ok=True)
joblib.dump(model, 'model/model.pkl')          # joblib, as the lecture used -- not pickle

reloaded = joblib.load('model/model.pkl')
check = float(reloaded.predict([[0.5, -0.2]])[0])
print(f'saved and reloaded; prediction for [0.5, -0.2] = {check:.4f}')
print('file exists:', os.path.exists('model/model.pkl'))

saved and reloaded; prediction for [0.5, -0.2] = 29.0467
file exists: True


## 2. Write `app.py`

The model is loaded **once, at import time**, next to the imports — not inside a route. A route that
called `joblib.load` on every request would re-read the file from disk for every caller, which is
the mistake this task exists to teach.

In [3]:
app_code = '''"""Flask API serving the Stage 13 homework model."""

import logging

import joblib
import numpy as np
from flask import Flask, jsonify, request

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("hw13")

# Loaded ONCE at startup, not per request.
MODEL = joblib.load("model/model.pkl")
N_FEATURES = 2

app = Flask(__name__)


def _predict(values):
    return float(MODEL.predict(np.array(values, dtype=float).reshape(1, -1))[0])


@app.post("/predict")
def predict_post():
    """Score a JSON body of the form {"features": [f1, f2]}."""
    body = request.get_json(silent=True)
    if not isinstance(body, dict) or "features" not in body:
        return jsonify({"error": "body must be JSON with a 'features' key"}), 400

    features = body["features"]
    if not isinstance(features, (list, tuple)) or len(features) != N_FEATURES:
        return jsonify(
            {"error": f"'features' must be a list of {N_FEATURES} numbers, got {features!r}"}
        ), 400
    try:
        values = [float(v) for v in features]
    except (TypeError, ValueError):
        return jsonify({"error": "every feature must be a number"}), 400

    prediction = _predict(values)
    logger.info("POST /predict %s -> %.4f", values, prediction)
    return jsonify({"prediction": prediction})


@app.get("/predict/<f1>/<f2>")
def predict_path(f1, f2):
    """Score two path parameters, for a person or a browser."""
    try:
        values = [float(f1), float(f2)]
    except ValueError:
        return jsonify({"error": f"both parameters must be numbers, got {f1!r} and {f2!r}"}), 400

    prediction = _predict(values)
    logger.info("GET /predict/%s/%s -> %.4f", f1, f2, prediction)
    return jsonify({"prediction": prediction})


if __name__ == "__main__":
    # 5002, not 5000: macOS Control Center binds 5000 and answers 403.
    app.run(host="127.0.0.1", port=5002, debug=False)
'''

with open('app.py', 'w') as fh:
    fh.write(app_code)
print(f'wrote app.py ({len(app_code):,} characters)')

wrote app.py (1,937 characters)


## 3. Launch the server

Started here in a background thread on an ephemeral port so this notebook runs top to bottom
unattended. `python app.py` starts the same app on port 5002 from a terminal.

In [4]:
from werkzeug.serving import make_server

import app as hw13_app

server = make_server('127.0.0.1', 0, hw13_app.app)     # port 0 -> the OS picks a free one
port = server.server_port
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()
BASE = f'http://127.0.0.1:{port}'
print('server running at', BASE)

server running at http://127.0.0.1:64295


## 4. Call the API

Both routes, plus one deliberately bad call. This output is the testing evidence.

In [5]:
r1 = requests.post(f'{BASE}/predict', json={'features': [0.5, -0.2]}, timeout=10)
print(f'POST /predict  {{"features": [0.5, -0.2]}}  -> HTTP {r1.status_code}')
print(json.dumps(r1.json(), indent=2))

POST /predict  {"features": [0.5, -0.2]}  -> HTTP 200
{
  "prediction": 29.046691402929017
}


In [6]:
r2 = requests.get(f'{BASE}/predict/0.5/-0.2', timeout=10)
print(f'GET  /predict/0.5/-0.2  -> HTTP {r2.status_code}')
print(json.dumps(r2.json(), indent=2))
print(f'\nboth routes agree: {abs(r1.json()["prediction"] - r2.json()["prediction"]) < 1e-9}')

GET  /predict/0.5/-0.2  -> HTTP 200
{
  "prediction": 29.046691402929017
}

both routes agree: True


In [7]:
bad = [
    ('POST, missing key', requests.post(f'{BASE}/predict', json={'x': [1, 2]}, timeout=10)),
    ('POST, wrong count', requests.post(f'{BASE}/predict', json={'features': [1.0]}, timeout=10)),
    ('GET, non-numeric  ', requests.get(f'{BASE}/predict/abc/0.2', timeout=10)),
]
for label, resp in bad:
    print(f'{label}  -> HTTP {resp.status_code}   {resp.json()}')

POST, missing key  -> HTTP 400   {'error': "body must be JSON with a 'features' key"}
POST, wrong count  -> HTTP 400   {'error': "'features' must be a list of 2 numbers, got [1.0]"}
GET, non-numeric    -> HTTP 400   {'error': "both parameters must be numbers, got 'abc' and '0.2'"}


In [8]:
server.shutdown()
thread.join(timeout=5)
print('server stopped')

server stopped


## 5. Write `README.md`

In [9]:
example = round(r1.json()["prediction"], 4)
readme = f"""# Homework 13 — Productization

A linear regression trained on 100 synthetic samples with two features
(`make_regression(n_samples=100, n_features=2, noise=0.1, random_state=42)`).
It takes two numbers and returns a single predicted value.

## Start the server

```bash
python app.py
```

Serves on `http://127.0.0.1:5002`. Port 5002 rather than 5000, because macOS Control Center binds
5000 and answers 403 — which looks like a bug in the app and is not.

## Routes

**POST /predict** — for a program sending JSON.

```bash
curl -s -X POST http://127.0.0.1:5002/predict \\
  -H 'Content-Type: application/json' \\
  -d '{{"features": [0.5, -0.2]}}'
```
```json
{{"prediction": {example}}}
```

**GET /predict/<f1>/<f2>** — for a person or a browser.

```bash
curl -s http://127.0.0.1:5002/predict/0.5/-0.2
```
```json
{{"prediction": {example}}}
```

Both routes use the same model object, loaded once when the app starts, so they return the same
number for the same input.

## Bad input

Both routes answer with JSON and HTTP 400 rather than a traceback:

| Request | Response |
|---|---|
| `{{"x": [1, 2]}}` — no `features` key | 400 `{{"error": "body must be JSON with a 'features' key"}}` |
| `{{"features": [1.0]}}` — wrong count | 400 `{{"error": "'features' must be a list of 2 numbers, got [1.0]"}}` |
| `/predict/abc/0.2` — not a number | 400 `{{"error": "both parameters must be numbers, got 'abc' and '0.2'"}}` |
"""

with open('README.md', 'w') as fh:
    fh.write(readme)
print(f'wrote README.md ({len(readme):,} characters)')
print('\nfolder contents:')
for path in sorted(os.listdir('.')):
    if not path.startswith('.'):
        print('  ', path)

wrote README.md (1,403 characters)

folder contents:
   README.md
   __pycache__
   app.py
   homework13_productization_submission.ipynb
   model


## What this homework was actually about

Training the model was the easy part and took three lines. The work was everything around it: saving
it so another process can load it, loading it **once** rather than per request, answering bad input
with a clear 400 instead of a stack trace, and writing a README that lets someone who has never seen
this folder call the thing.

That last point is the one worth keeping. An API nobody can work out how to call is not finished.